# 🤗 Hugging Face — Publicando seu Modelo no Hub
**FIAP Postech — IA para Devs | Fase 1 · Fundamentos de IA e Machine Learning**

---

## O que você vai aprender
1. O que é o Hugging Face Hub e por que usá-lo
2. Criar e configurar sua conta + token de acesso
3. Salvar um modelo sklearn no formato correto
4. Criar o `model card` (README) — obrigatório para boas práticas
5. Fazer upload do modelo para o Hub
6. Carregar e usar o modelo publicado (como qualquer pessoa faria)
7. Publicar um pipeline sklearn como módulo reutilizável

---
> **Por que o Hugging Face?**  
> É o GitHub dos modelos de ML. Permite versioning, colaboração, e qualquer pessoa do mundo pode baixar e usar seu modelo com 2 linhas de código.

---
## 0. Instalação

In [ ]:
!pip install huggingface_hub scikit-learn joblib --quiet
print('✅ Dependências instaladas')

In [ ]:
import numpy as np
import pandas as pd
import joblib
import json
import os
from pathlib import Path

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

from huggingface_hub import HfApi, HfFolder, login, Repository

print('✅ Imports OK')

---
## 1. Configuração do Hugging Face

### Passo a passo para criar sua conta e token:

1. Acesse [huggingface.co](https://huggingface.co) e crie uma conta gratuita
2. Vá em **Settings → Access Tokens → New Token**
3. Tipo: `write` (para fazer upload)
4. Copie o token e cole abaixo

> ⚠️ **NUNCA** commite seu token no Git. Use variável de ambiente ou arquivo `.env`.

In [ ]:
# Opção 1: Login interativo (recomendado — pede o token de forma segura)
login()  # vai pedir o token via input seguro

# Opção 2: Via variável de ambiente (para uso em CI/CD)
# import os
# login(token=os.environ['HF_TOKEN'])

---
## 2. Treinando o Modelo

In [ ]:
# Treinando o pipeline completo
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_proba)
print('📊 Métricas do modelo:')
print(classification_report(y_test, y_pred, target_names=data.target_names))
print(f'AUC Score: {auc:.4f}')

---
## 3. Preparando os Arquivos para Upload

O Hugging Face Hub espera uma estrutura de repositório. Vamos criar:

In [ ]:
# ⚠️ ALTERE PARA O SEU USUÁRIO DO HUGGING FACE
HF_USERNAME = 'seu-usuario-aqui'  # ex: 'joaosilva'
REPO_NAME = 'breast-cancer-classifier-fiap'
REPO_ID = f'{HF_USERNAME}/{REPO_NAME}'

# Criando diretório local do modelo
model_dir = Path('./modelo_hf')
model_dir.mkdir(exist_ok=True)
print(f'📁 Diretório: {model_dir}')

In [ ]:
# 3.1 Salvando o pipeline sklearn com joblib
model_path = model_dir / 'model.joblib'
joblib.dump(pipeline, model_path)
print(f'✅ Modelo salvo: {model_path} ({model_path.stat().st_size / 1024:.1f} KB)')

In [ ]:
# 3.2 Salvando metadados do modelo (config.json)
from sklearn.metrics import recall_score, f1_score, accuracy_score

config = {
    'model_type': 'sklearn_pipeline',
    'algorithm': 'RandomForestClassifier',
    'task': 'binary_classification',
    'dataset': 'Wisconsin Breast Cancer (sklearn)',
    'n_features': len(data.feature_names),
    'feature_names': list(data.feature_names),
    'classes': list(data.target_names),
    'hyperparameters': {
        'n_estimators': 100,
        'random_state': 42
    },
    'metrics': {
        'accuracy': accuracy_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'auc_roc': auc
    },
    'train_size': len(X_train),
    'test_size': len(X_test),
    'sklearn_version': __import__('sklearn').__version__
}

config_path = model_dir / 'config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print('✅ config.json salvo:')
print(json.dumps(config, indent=2))

In [ ]:
# 3.3 Criando o Model Card (README.md)
# O model card é a "documentação" do seu modelo — OBRIGATÓRIO para boas práticas

model_card = f"""---
license: mit
tags:
- sklearn
- classification
- healthcare
- breast-cancer
- random-forest
metrics:
- accuracy
- f1
- roc_auc
language:
- pt
---

# 🩺 Breast Cancer Classifier — FIAP Postech IA para Devs

Modelo de classificação binária para predição de tumores malignos/benignos 
treinado no dataset Wisconsin Breast Cancer.

Desenvolvido como projeto acadêmico para a Pós-Graduação **IA para Devs** da FIAP Postech.

## Sobre o Modelo

| Atributo | Valor |
|---|---|
| Algoritmo | Random Forest Classifier |
| Pré-processamento | StandardScaler |
| Dataset | Wisconsin Breast Cancer (sklearn) |
| Features | 30 features numéricas |
| Classes | malignant (0), benign (1) |

## Métricas de Desempenho

| Métrica | Valor |
|---|---|
| Accuracy | {config['metrics']['accuracy']:.4f} |
| Recall | {config['metrics']['recall']:.4f} |
| F1 Score | {config['metrics']['f1_score']:.4f} |
| AUC-ROC | {config['metrics']['auc_roc']:.4f} |

> **Métrica principal: Recall** — em diagnóstico oncológico, minimizamos falsos negativos.

## Como Usar

```python
from huggingface_hub import hf_hub_download
import joblib

# Download do modelo
model_path = hf_hub_download(
    repo_id='{REPO_ID}',
    filename='model.joblib'
)
model = joblib.load(model_path)

# Predição
# X deve ter as 30 features do dataset Wisconsin Breast Cancer
predicao = model.predict(X_novo)
probabilidade = model.predict_proba(X_novo)[:, 1]
```

## Features Esperadas

O modelo espera as 30 features numéricas do Wisconsin Breast Cancer:
{', '.join(data.feature_names)}

## Limitações e Avisos Éticos

⚠️ Este modelo é **apenas para fins acadêmicos**. Não deve ser usado para diagnóstico clínico real.

## Contato

Desenvolvido por: {HF_USERNAME}  
Curso: FIAP Postech — IA para Devs  
"""

readme_path = model_dir / 'README.md'
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(model_card)

print('✅ README.md (Model Card) salvo')
print('\n📁 Arquivos prontos para upload:')
for f in model_dir.iterdir():
    print(f'   {f.name} ({f.stat().st_size / 1024:.1f} KB)')

---
## 4. Upload para o Hugging Face Hub

In [ ]:
api = HfApi()

# Criando o repositório no Hub (se não existir)
try:
    api.create_repo(
        repo_id=REPO_ID,
        repo_type='model',
        private=False,  # True para privado
        exist_ok=True   # não dá erro se já existir
    )
    print(f'✅ Repositório criado/verificado: https://huggingface.co/{REPO_ID}')
except Exception as e:
    print(f'❌ Erro ao criar repositório: {e}')

In [ ]:
# Upload de todos os arquivos
arquivos = ['model.joblib', 'config.json', 'README.md']

for arquivo in arquivos:
    local_path = model_dir / arquivo
    try:
        api.upload_file(
            path_or_fileobj=str(local_path),
            path_in_repo=arquivo,
            repo_id=REPO_ID,
            repo_type='model',
            commit_message=f'Add {arquivo}'
        )
        print(f'✅ {arquivo} enviado')
    except Exception as e:
        print(f'❌ Erro no upload de {arquivo}: {e}')

print(f'\n🎉 Modelo publicado em: https://huggingface.co/{REPO_ID}')

---
## 5. Carregando o Modelo Publicado
Simulando como qualquer pessoa faria para usar seu modelo.

In [ ]:
from huggingface_hub import hf_hub_download

# Download do modelo do Hub
downloaded_model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename='model.joblib'
)

# Carregando
modelo_carregado = joblib.load(downloaded_model_path)
print(f'✅ Modelo carregado de: {REPO_ID}')
print(f'   Tipo: {type(modelo_carregado)}')
print(f'   Steps: {list(modelo_carregado.named_steps.keys())}')

In [ ]:
# Testando o modelo carregado
y_pred_hf = modelo_carregado.predict(X_test)
y_proba_hf = modelo_carregado.predict_proba(X_test)[:, 1]

print('📊 Predição com modelo baixado do Hub:')
print(classification_report(y_test, y_pred_hf, target_names=data.target_names))
print(f'AUC: {roc_auc_score(y_test, y_proba_hf):.4f}')

# Predição em uma amostra nova
amostra = X_test.iloc[[0]]
pred = modelo_carregado.predict(amostra)[0]
prob = modelo_carregado.predict_proba(amostra)[0]
print(f'\n🔍 Exemplo de predição:')
print(f'   Predição: {data.target_names[pred]} ({["malignant", "benign"][pred]})')
print(f'   Probabilidade: malignant={prob[0]:.3f}, benign={prob[1]:.3f}')

---
## 6. Carregando os Metadados

In [ ]:
config_path_hf = hf_hub_download(repo_id=REPO_ID, filename='config.json')

with open(config_path_hf) as f:
    config_carregado = json.load(f)

print('📋 Metadados do modelo:')
print(f'   Algoritmo: {config_carregado["algorithm"]}')
print(f'   Features: {config_carregado["n_features"]}')
print(f'   Métricas registradas:')
for k, v in config_carregado['metrics'].items():
    print(f'     {k}: {v:.4f}')

---
## 7. Extras — Boas Práticas

### Versionamento
Cada upload cria um commit no repositório. Para versionar explicitamente:

```python
api.upload_file(
    ...,
    commit_message='v1.1 — adicionado Gradient Boosting, recall melhorou 2%'
)
```

### Buscar modelos do Hub (uso avançado)

```python
from huggingface_hub import list_models

# Listar modelos de classificação com sklearn
modelos = list(list_models(filter='sklearn', task='tabular-classification'))
for m in modelos[:5]:
    print(m.id, m.downloads)
```

### Dataset no Hub

```python
api.create_repo(repo_id='meu-usuario/meu-dataset', repo_type='dataset')
api.upload_file(path_or_fileobj='data.csv', path_in_repo='data.csv',
                repo_id='meu-usuario/meu-dataset', repo_type='dataset')
```

---
## Resumo do Fluxo

```
Treinar modelo → Salvar com joblib → Criar config.json + README.md
     ↓
api.create_repo() → api.upload_file() × 3
     ↓
https://huggingface.co/seu-usuario/seu-modelo
     ↓
Qualquer pessoa: hf_hub_download() → joblib.load() → .predict()
```